In [1]:
import pandas as pd
import numpy as np
import os
import json
from scipy import constants
from pysr import PySRRegressor
import matplotlib.pyplot as plt
import sympy
import random
from ase.io import read
from pymatgen.io.ase import AseAtomsAdaptor

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [2]:
old_dataset = pd.read_pickle('/home/paolo/Na_dataset/diffusion.pckl.gzip', compression='gzip')

In [3]:
atoms_train = read("/home/paolo/Na_dataset/train_Na_new.xyz", index=":")
atoms_test = read("/home/paolo/Na_dataset/test_Na_new.xyz", index=":")

In [4]:
data_train = pd.concat(
    [pd.DataFrame([a.info for a in atoms_train]),
     pd.DataFrame({'volume': [a.get_volume() for a in atoms_train]})],
    axis=1
)
data_test = pd.concat(
    [pd.DataFrame([a.info for a in atoms_test]),
     pd.DataFrame({'volume': [a.get_volume() for a in atoms_test]})],
    axis=1
)

In [5]:
len(set(old_dataset['VPA']).intersection(data_train['VPA']))

1041

Realigns the sequences

In [6]:
index_map = {val: idx for idx, val in enumerate(old_dataset['VPA'])}
matching_indices = data_train['VPA'].map(index_map)
aligned_VPA = old_dataset['VPA'].iloc[matching_indices]
print(aligned_VPA)

479     23.177308
725     30.285038
312     44.860599
853     42.465200
1277    34.721477
          ...    
180     42.526695
301     27.705721
497     23.954086
1127    32.081979
1081    31.092229
Name: VPA, Length: 1041, dtype: float64


In [13]:
old_dataset['I%'].iloc[matching_indices].reset_index(drop=True) == data_train['I']

0       True
1       True
2       True
3       True
4       True
        ... 
1036    True
1037    True
1038    True
1039    True
1040    True
Length: 1041, dtype: bool

In [14]:
old_dataset['Ea'].iloc[matching_indices].reset_index(drop=True)

0       0.364992
1       0.425423
2       0.274549
3       0.325011
4       0.344974
          ...   
1036    0.321702
1037     0.39034
1038    0.392994
1039    0.336112
1040    0.449709
Name: Ea, Length: 1041, dtype: object

In [15]:
new_temps_train = pd.DataFrame({f'logD_{i+1}': np.log(old_dataset['D_sim'].str[i].iloc[matching_indices]) for i in range(4)}).reset_index(drop=True)
new_temps_test = pd.DataFrame({f'logD_{i+1}': np.log(old_dataset['D_sim'].str[i].iloc[~matching_indices]) for i in range(4)}).reset_index(drop=True)

In [17]:
data_train_temps = pd.concat(
    [pd.DataFrame({'Ea_old': old_dataset['Ea'].iloc[matching_indices].reset_index(drop=True)}),
        data_train, new_temps_train], axis=1
)
data_test_temps = pd.concat(
    [pd.DataFrame({'Ea_old': old_dataset['Ea'].iloc[~matching_indices].reset_index(drop=True)}),
        data_test, new_temps_test],
    axis=1
)

In [18]:
data_train_temps

,Ea_old,Ea,B,N,O,F,Na,Al,Si,P,...,pockets-per-ion,pockets-per-volume,paths-per-ion,paths-per-volume,poly_dist_avg_fixed,volume,logD_1,logD_2,logD_3,logD_4
0,0.364992,0.365113,0.000000,0.0,0.631387,0.000000,0.127737,0.000000,0.000000,0.171533,...,5.371429,0.046887,45.342857,0.395793,0.385197,16038.697093,-16.619045,-14.792911,-12.941451,-12.037079
1,0.425423,0.425024,0.000000,0.0,0.467925,0.000000,0.184906,0.000000,0.000000,0.135849,...,1.173469,0.011507,5.484694,0.053782,0.375974,19988.125341,-16.330785,-13.792352,-11.983093,-10.964065
2,0.274549,0.274638,0.113043,0.0,0.000000,0.000000,0.208696,0.108696,0.000000,0.000000,...,2.583333,0.021100,14.566667,0.118978,0.080752,29383.692649,-13.305840,-11.465448,-10.487184,-9.819893
3,0.325011,0.324737,0.000000,0.0,0.000000,0.000000,0.101124,0.000000,0.243446,0.033708,...,8.425926,0.032273,51.564815,0.197504,0.116010,28196.892709,-14.102597,-12.112593,-10.741215,-10.017241
4,0.344974,0.345059,0.000000,0.0,0.407692,0.000000,0.219231,0.000000,0.000000,0.161538,...,2.105263,0.021466,13.359649,0.136222,0.363843,22360.631045,-14.593970,-12.548051,-11.006865,-10.278066
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1036,0.321702,0.321759,0.000000,0.0,0.000000,0.000000,0.241935,0.000000,0.000000,0.161290,...,1.360784,0.012972,5.980392,0.057011,0.091131,26749.291087,-13.705843,-11.505902,-10.276437,-9.685011
1037,0.39034,0.389788,0.000000,0.0,0.380711,0.106599,0.335025,0.177665,0.000000,0.000000,...,0.335859,0.008334,2.255051,0.055958,0.317510,15958.495344,-16.880391,-14.715289,-12.954074,-11.945440
1038,0.392994,0.393305,0.250000,0.0,0.500000,0.000000,0.250000,0.000000,0.000000,0.000000,...,0.412000,0.008600,1.924000,0.040160,0.368284,11977.043019,-16.563492,-14.078893,-12.544274,-11.587029
1039,0.336112,0.335168,0.000000,0.0,0.417969,0.000000,0.187500,0.000000,0.000000,0.027344,...,3.026042,0.029022,16.453125,0.157799,0.301010,20019.155151,-14.815149,-12.694010,-11.391324,-10.551972


In [19]:
np.log(old_dataset['D_sim'].iloc[matching_indices].str[0]).reset_index(drop=True) == data_train_temps['logD_1']

0       True
1       True
2       True
3       True
4       True
        ... 
1036    True
1037    True
1038    True
1039    True
1040    True
Length: 1041, dtype: bool

In [23]:
def initialize(df, med_flag = False):
    kb = constants.physical_constants['Boltzmann constant in eV/K'][0]
    ln_D_all = np.array(df['logD'])
    if (med_flag == True):
        med =  np.median(ln_D_all).round()
        ln_D = ln_D_all[ln_D_all>med]
        ind = np.where(ln_D_all > med)[0]
    else:
        med = -100
        ln_D = ln_D_all[ln_D_all>med]
        ind = np.where(ln_D_all > med)[0]
    
    return {
        "ln_D_all" : ln_D_all,
        "ln_D_300" : ln_D,
        "ln_D_500" : np.array(df['logD_1'], dtype='float')[ind],
        "ln_D_700" : np.array(df['logD_2'], dtype='float')[ind],
        "ln_D_900" : np.array(df['logD_3'], dtype='float')[ind],
        "ln_D_1100" : np.array(df['logD_4'], dtype='float')[ind],
        
        "E" : np.array(df['Ea'], dtype = 'float')[ind],
        "E_old" : np.array(df['Ea_old'], dtype = 'float')[ind],
        "V" : np.array(df['volume'], dtype = 'float')[ind],
        
        # Atomic fractions of a given element
        "B":  np.array(df['B'], dtype='float')[ind],
        "N":  np.array(df['N'], dtype='float')[ind],
        "O":  np.array(df['O'], dtype='float')[ind],
        "F":  np.array(df['F'], dtype='float')[ind],
        "Na": np.array(df['Na'], dtype='float')[ind],
        "Al": np.array(df['Al'], dtype='float')[ind],
        "Si": np.array(df['Si'], dtype='float')[ind],
        "P":  np.array(df['P'], dtype='float')[ind],
        "S":  np.array(df['S'], dtype='float')[ind],
        "Cl": np.array(df['Cl'], dtype='float')[ind],
        "Ge": np.array(df['Ge'], dtype='float')[ind],
        "Br": np.array(df['Br'], dtype='float')[ind],
        "I":  np.array(df['I'], dtype='float')[ind],
        
        "density": np.array(df['density'], dtype='float')[ind],
        # Average Atomic Volume
        "AAV": np.array(df['AAV'], dtype='float')[ind],
        # Packing fraction
        "PF": np.array(df['PF'], dtype='float')[ind],
        # set-of-non-Na-atoms bond ionicity
        "SBI": np.array(df['SBI'], dtype='float')[ind],
        # Volume per anion
        "VPA": np.array(df['VPA'], dtype='float')[ind],
        # Na bond ionicity
        "NBI": np.array(df['NBI'], dtype='float')[ind],
        # Ratio of NBI and SBI
        "RBI": np.array(df['RBI'], dtype='float')[ind],
        # Weighted electronegativity
        "WE": np.array(df['weighted_electronegativity'], dtype='float')[ind],
        # -Na-Na (are just the Na-Na distances)
        "Na-Na": np.array(df['Na-Na'], dtype='float')[ind],
        "Na-Na-max": np.array(df['Na-Na-max'], dtype='float')[ind],
        "Na-Na-avg": np.array(df['Na-Na-avg'], dtype='float')[ind],
        # -bottleneck (free distance available for a Na⁺ ion to pass through the narrowest part of a diffusion pathway formed by surrounding framework atoms (typically O²⁻, S²⁻, or halides))
        "bottleneck_min": np.array(df['bottleneck_min'], dtype='float')[ind],
        "bottleneck_avg": np.array(df['bottleneck_avg'], dtype='float')[ind],
        # -poly_dist (Polyhedral distortion is the degree to which the bond lengths and bond angles of a coordination polyhedron differ from those of an ideal reference polyhedron.)
        "poly_dist_max": np.array(df['poly_dist_max'], dtype='float')[ind],
        "poly_dist_std": np.array(df['poly_dist_std'], dtype='float')[ind],
        "poly_dist_avg": np.array(df['poly_dist_avg'], dtype='float')[ind],
        "poly_dist_avg_fixed": np.array(df['poly_dist_avg_fixed'], dtype='float')[ind],
        # -face-area   (area of the shared face between two neighboring Voronoi polyhedra corresponding to adjacent Na sites it represents the cross-sectional area available for Na⁺ migration between those sites)
        "face-area-max": np.array(df['face-area-max'], dtype='float')[ind],
        "face-area-avg": np.array(df['face-area-avg'], dtype='float')[ind],
        "face-area-min": np.array(df['face-area-min'], dtype='float')[ind],
        "face-area-count": np.array(df['face-area-count'], dtype='float')[ind],
        # -pockets  (free-volume region in the crystal structure where a Na⁺ ion can reside with relatively low energy, typically corresponding to a stable or metastable Na site)
        "pockets-per-ion": np.array(df['pockets-per-ion'], dtype='float')[ind],
        "pockets-per-volume": np.array(df['pockets-per-volume'], dtype='float')[ind],
        # -paths  (connected sequence of pockets linked by Voronoi faces and constrained by bottlenecks, forming a continuous route that allows Na⁺ ions to move through the crystal lattice)
        "paths-per-ion": np.array(df['paths-per-ion'], dtype='float')[ind],
        "paths-per-volume": np.array(df['paths-per-volume'], dtype='float')[ind],
    }

In [37]:
kb = constants.physical_constants['Boltzmann constant in eV/K'][0]
data = initialize(data_train_temps)
T = 1100
lnDvsE = pd.DataFrame({
    "sample_id": [f"s{j+1}" for j in range(len(data['ln_D_300']))], 
    "E_T": data['E']/(kb*T),
    #"E_T_old": data['E_old']/(kb*T),
    "rho": data['density'],
    "AAV": data['AAV'],
    "PF": data['PF'],
    "VPA": data['VPA'],
    "NBI": data['NBI'],
    "SBI": data['SBI'],
    "RBI": data['RBI'],
    "WE": data['WE'],
    "B": data['B'],
    "N": data['N'],
    "S": data['S'],
    "O": data['O'],
    "F": data['F'],
    "Na": data['Na'],
    "Al": data['Al'],
    "Si": data['Si'],
    "P": data['P'],
    "Cl": data['Cl'],
    "Ge": data['Ge'],
    "Br": data['Br'],
    "I": data['I'],

    # Volume
    "cub_root_V" : data['V']**(1/3),
    
    # Na–Na distances
    "Na-Na": data['Na-Na'],
    "Na-Na-max": data['Na-Na-max'],
    "Na-Na-avg": data['Na-Na-avg'],

    # Bottleneck descriptors
    "bottleneck_min": data['bottleneck_min'],
    "bottleneck_avg": data['bottleneck_avg'],

    # Polyhedral distortion
    "poly_dist_max": data['poly_dist_max'],
    "poly_dist_std": data['poly_dist_std'],
    "poly_dist_avg": data['poly_dist_avg'],
    "poly_dist_avg_fixed": data['poly_dist_avg_fixed'],

    # Voronoi face areas
    "face-area-max": data['face-area-max'],
    "face-area-avg": data['face-area-avg'],
    "face-area-min": data['face-area-min'],
    "face-area-count": data['face-area-count'],

    # Pockets
    "pockets-per-ion": data['pockets-per-ion'],
    "pockets-per-volume": data['pockets-per-volume'],

    # Paths
    "paths-per-ion": data['paths-per-ion'],
    "paths-per-volume": data['paths-per-volume'],

    # Target
    f"ln_D_{T}": data[f'ln_D_{T}']
})

In [38]:
n_feat = 40
lnDvsE.to_csv(f"/home/paolo/Thesis/Na/data/dataset2/Na_lndiff{n_feat}_temp{T}.csv", index=False)
data_file = f"/home/paolo/Thesis/Na/data/dataset2/Na_lndiff{n_feat}_temp{T}.csv"
os.environ["SISSOPP_TUT_DAT_FILE"] = data_file

In [164]:
# # Try the fit with the exact functional form
# rung = 2
# Diff_json = {
#     'data_file': data_file,
#      'property_key': 'ln_D_500',
#      'desc_dim': 1,
#      'n_sis_select': 10,
#      'max_rung': rung,
#      'calc_type': 'regression',
#      'min_abs_feat_val': 1e-05,
#      'max_abs_feat_val': 1e8,
#      'n_residual': 10,
#      'n_models_store': 10,
#      'leave_out_frac': 0.0,
#      'leave_out_inds': [],
#      'opset': ['exp', 'add', 'sub', 'mult', 'div', 'sin', 'cos']
# }
# json.dump(Diff_json, open(f"/home/paolo/Thesis/Na/data/dataset2/lndiff{n_feat}_{rung}_temp500.json", "w"), indent=4)

In [ ]:
# Rung=2 Features=40 'E_T': E/(kb*T), 'rho': density, 'AAV': AAV, 'PF': PF, 'VPA': VPA, 'NBI': NBI, 'SBI': SBI,  'RBI': RBI, 'WE': WE, 
        # 'B': B, 'N': N, 'S': S, 'O': O, 'F': F, 'Na': Na, 'Al': Al, 'Si': Si, 'P': P, 'Cl': Cl, 'Ge': Ge, 'Br': Br, 'I': I, 'cub_root_V': V^(1/3),
        # "Na-Na": data['Na-Na'], "Na-Na-max": data['Na-Na-max'], "Na-Na-avg": data['Na-Na-avg'], "bottleneck_min": data['bottleneck_min'], "bottleneck_avg": data['bottleneck_avg'],
        # "poly_dist_max": data['poly_dist_max'], "poly_dist_std": data['poly_dist_std'], "poly_dist_avg": data['poly_dist_avg'], "poly_dist_avg_fixed": data['poly_dist_avg_fixed'],
        # "face-area-max": data['face-area-max'], "face-area-avg": data['face-area-avg'], "face-area-min": data['face-area-min'], "face-area-count": data['face-area-count'],
        # "pockets-per-ion": data['pockets-per-ion'], "pockets-per-volume": data['pockets-per-volume'],
        # "paths-per-ion": data['paths-per-ion'], "paths-per-volume": data['paths-per-volume'],
# T = 500K
# time input_parsing: 0.0627144 s
# Time to generate feat space: 166.804 s
# Projection time: 7.70674 s
# Time to get best features on rank : 0.0001358 s
# |---Complete final combination/selection from all ranks: 0.0070805 s
# Time for SIS: 11.2042 s
# Time for l0-norm: 0.0050608 s
# Train RMSE: 0.374456 Unitless
# c0 + a0 * ((face-area-avg - O) / (face-area-avg + E_T))

In [39]:
# Try the fit with the exact functional form
rung = 1
desc_dim = 3
Diff_json = {
    'data_file': data_file,
     'property_key': f'ln_D_{T}',
     'desc_dim': desc_dim,
     'n_sis_select': 10,
     'max_rung': rung,
     'calc_type': 'regression',
     'min_abs_feat_val': 1e-05,
     'max_abs_feat_val': 1e8,
     'n_residual': 10,
     'n_models_store': 10,
     'leave_out_frac': 0.0,
     'leave_out_inds': [],
     'opset': ['exp', 'add', 'sub', 'mult', 'div', 'sin', 'cos']
}
json.dump(Diff_json, open(f"/home/paolo/Thesis/Na/data/desc_dim/lndiff{n_feat}_{rung}_{desc_dim}_temp{T}.json", "w"), indent=4)

In [ ]:
# Rung=1 desc_dim=2 Features=40 'E_T': E/(kb*T), 'rho': density, 'AAV': AAV, 'PF': PF, 'VPA': VPA, 'NBI': NBI, 'SBI': SBI,  'RBI': RBI, 'WE': WE, 
        # 'B': B, 'N': N, 'S': S, 'O': O, 'F': F, 'Na': Na, 'Al': Al, 'Si': Si, 'P': P, 'Cl': Cl, 'Ge': Ge, 'Br': Br, 'I': I, 'cub_root_V': V^(1/3),
        # "Na-Na": data['Na-Na'], "Na-Na-max": data['Na-Na-max'], "Na-Na-avg": data['Na-Na-avg'], "bottleneck_min": data['bottleneck_min'], "bottleneck_avg": data['bottleneck_avg'],
        # "poly_dist_max": data['poly_dist_max'], "poly_dist_std": data['poly_dist_std'], "poly_dist_avg": data['poly_dist_avg'], "poly_dist_avg_fixed": data['poly_dist_avg_fixed'],
        # "face-area-max": data['face-area-max'], "face-area-avg": data['face-area-avg'], "face-area-min": data['face-area-min'], "face-area-count": data['face-area-count'],
        # "pockets-per-ion": data['pockets-per-ion'], "pockets-per-volume": data['pockets-per-volume'],
        # "paths-per-ion": data['paths-per-ion'], "paths-per-volume": data['paths-per-volume'],
# T = 500K
# time input_parsing: 0.04308 s
# Time to generate feat space: 0.0218247 s
# Projection time: 0.0176422 s
# Time to get best features on rank : 9.36e-05 s
# |---Complete final combination/selection from all ranks: 0.0004002 s
# Time for SIS: 0.0187529 s
# Time for l0-norm: 0.0190608 s
# Projection time: 0.0090296 s
# Time to get best features on rank : 7.64e-05 s
# |---Complete final combination/selection from all ranks: 0.0002922 s
# Time for SIS: 0.0109572 s
# Time for l0-norm: 0.0051133 s
# Train RMSE: 0.472973 Unitless
# c0 + a0 * (cub_root_V - E_T)

# Train RMSE: 0.355972 Unitless
# c0 + a0 * (Al + O) + a1 * (face-area-avg / E_T)
# Coefficients
# Task   a0                      a1                      c0
# all , -1.924301202233034e+00,  7.571736186805641e+00, -1.742091065463907e+01, 

In [ ]:
# Rung=1 desc_dim=2 Features=40 'E_T': E/(kb*T), 'rho': density, 'AAV': AAV, 'PF': PF, 'VPA': VPA, 'NBI': NBI, 'SBI': SBI,  'RBI': RBI, 'WE': WE, 
        # 'B': B, 'N': N, 'S': S, 'O': O, 'F': F, 'Na': Na, 'Al': Al, 'Si': Si, 'P': P, 'Cl': Cl, 'Ge': Ge, 'Br': Br, 'I': I, 'cub_root_V': V^(1/3),
        # "Na-Na": data['Na-Na'], "Na-Na-max": data['Na-Na-max'], "Na-Na-avg": data['Na-Na-avg'], "bottleneck_min": data['bottleneck_min'], "bottleneck_avg": data['bottleneck_avg'],
        # "poly_dist_max": data['poly_dist_max'], "poly_dist_std": data['poly_dist_std'], "poly_dist_avg": data['poly_dist_avg'], "poly_dist_avg_fixed": data['poly_dist_avg_fixed'],
        # "face-area-max": data['face-area-max'], "face-area-avg": data['face-area-avg'], "face-area-min": data['face-area-min'], "face-area-count": data['face-area-count'],
        # "pockets-per-ion": data['pockets-per-ion'], "pockets-per-volume": data['pockets-per-volume'],
        # "paths-per-ion": data['paths-per-ion'], "paths-per-volume": data['paths-per-volume'],
# T = 700K
# time input_parsing: 0.0529483 s
# Time to generate feat space: 0.0307294 s
# Projection time: 0.0146943 s
# Time to get best features on rank : 9.13e-05 s
# |---Complete final combination/selection from all ranks: 0.0004057 s
# Time for SIS: 0.0157973 s
# Time for l0-norm: 0.0265031 s
# Projection time: 0.0073726 s
# Time to get best features on rank : 7.71e-05 s
# |---Complete final combination/selection from all ranks: 0.0002979 s
# Time for SIS: 0.0094068 s
# Time for l0-norm: 0.0054594 s
# Train RMSE: 0.499792 Unitless
# c0 + a0 * (face-area-avg - O)

# Train RMSE: 0.419316 Unitless
# c0 + a0 * (poly_dist_avg_fixed - NBI) + a1 * (face-area-avg + B)
# Coefficients
# Task   a0                      a1                      c0
# all , -5.963001166541903e+00,  1.846998698607001e+00, -2.040026532965230e+01, 

In [ ]:
# Rung=1 desc_dim=3 Features=40 'E_T': E/(kb*T), 'rho': density, 'AAV': AAV, 'PF': PF, 'VPA': VPA, 'NBI': NBI, 'SBI': SBI,  'RBI': RBI, 'WE': WE, 
        # 'B': B, 'N': N, 'S': S, 'O': O, 'F': F, 'Na': Na, 'Al': Al, 'Si': Si, 'P': P, 'Cl': Cl, 'Ge': Ge, 'Br': Br, 'I': I, 'cub_root_V': V^(1/3),
        # "Na-Na": data['Na-Na'], "Na-Na-max": data['Na-Na-max'], "Na-Na-avg": data['Na-Na-avg'], "bottleneck_min": data['bottleneck_min'], "bottleneck_avg": data['bottleneck_avg'],
        # "poly_dist_max": data['poly_dist_max'], "poly_dist_std": data['poly_dist_std'], "poly_dist_avg": data['poly_dist_avg'], "poly_dist_avg_fixed": data['poly_dist_avg_fixed'],
        # "face-area-max": data['face-area-max'], "face-area-avg": data['face-area-avg'], "face-area-min": data['face-area-min'], "face-area-count": data['face-area-count'],
        # "pockets-per-ion": data['pockets-per-ion'], "pockets-per-volume": data['pockets-per-volume'],
        # "paths-per-ion": data['paths-per-ion'], "paths-per-volume": data['paths-per-volume'],
# T = 700K
# time input_parsing: 0.0435126 s
# Time to generate feat space: 0.0440128 s
# Projection time: 0.0028577 s
# Time to get best features on rank : 8.66e-05 s
# |---Complete final combination/selection from all ranks: 0.0004846 s
# Time for SIS: 0.0041027 s
# Time for l0-norm: 0.0050441 s
# Projection time: 0.0070322 s
# Time to get best features on rank : 7.61e-05 s
# |---Complete final combination/selection from all ranks: 0.0004006 s
# Time for SIS: 0.0090826 s
# Time for l0-norm: 0.0054144 s
# Projection time: 0.0047919 s
# Time to get best features on rank : 9.84e-05 s
# |---Complete final combination/selection from all ranks: 0.0037374 s
# Time for SIS: 0.0105226 s
# Time for l0-norm: 0.024643 s
# Train RMSE: 0.499792 Unitless
# c0 + a0 * (face-area-avg - O)

# Train RMSE: 0.419316 Unitless
# c0 + a0 * (poly_dist_avg_fixed - NBI) + a1 * (face-area-avg + B)

# Train RMSE: 0.373393 Unitless
# c0 + a0 * (P + B) + a1 * (poly_dist_avg_fixed - NBI) + a2 * (face-area-avg - O)
# Coefficients
# Task   a0                      a1                      a2                      c0
# all ,  2.661770859700893e+00, -4.346249156710553e+00,  1.460929854035220e+00, -1.858542978215206e+01,

In [ ]:
# Rung=1 desc_dim=3 Features=40 'E_T': E/(kb*T), 'rho': density, 'AAV': AAV, 'PF': PF, 'VPA': VPA, 'NBI': NBI, 'SBI': SBI,  'RBI': RBI, 'WE': WE, 
        # 'B': B, 'N': N, 'S': S, 'O': O, 'F': F, 'Na': Na, 'Al': Al, 'Si': Si, 'P': P, 'Cl': Cl, 'Ge': Ge, 'Br': Br, 'I': I, 'cub_root_V': V^(1/3),
        # "Na-Na": data['Na-Na'], "Na-Na-max": data['Na-Na-max'], "Na-Na-avg": data['Na-Na-avg'], "bottleneck_min": data['bottleneck_min'], "bottleneck_avg": data['bottleneck_avg'],
        # "poly_dist_max": data['poly_dist_max'], "poly_dist_std": data['poly_dist_std'], "poly_dist_avg": data['poly_dist_avg'], "poly_dist_avg_fixed": data['poly_dist_avg_fixed'],
        # "face-area-max": data['face-area-max'], "face-area-avg": data['face-area-avg'], "face-area-min": data['face-area-min'], "face-area-count": data['face-area-count'],
        # "pockets-per-ion": data['pockets-per-ion'], "pockets-per-volume": data['pockets-per-volume'],
        # "paths-per-ion": data['paths-per-ion'], "paths-per-volume": data['paths-per-volume'],
# T = 900K
# time input_parsing: 0.0457097 s
# Time to generate feat space: 0.021951 s
# Projection time: 0.0016003 s
# Time to get best features on rank : 8.39e-05 s
# |---Complete final combination/selection from all ranks: 0.0003794 s
# Time for SIS: 0.0026169 s
# Time for l0-norm: 0.0041017 s
# Projection time: 0.0070459 s
# Time to get best features on rank : 7.6e-05 s
# |---Complete final combination/selection from all ranks: 0.000296 s
# Time for SIS: 0.0090347 s
# Time for l0-norm: 0.0051009 s
# Projection time: 0.0051799 s
# Time to get best features on rank : 8.68e-05 s
# |---Complete final combination/selection from all ranks: 0.0003006 s
# Time for SIS: 0.0074284 s
# Time for l0-norm: 0.0239484 s
# Train RMSE: 0.434599 Unitless
# c0 + a0 * (face-area-avg - O)

# Train RMSE: 0.351456 Unitless
# c0 + a0 * (Al - B) + a1 * (face-area-avg - O)

# Train RMSE: 0.298775 Unitless
# c0 + a0 * (NBI - rho) + a1 * (Si + Al) + a2 * (face-area-avg - O)
# Coefficients
# Task   a0                      a1                      a2                      c0
# all ,  6.532163607574086e-01, -2.894702802801289e+00,  1.296854352349580e+00, -1.361343655316543e+01,

In [ ]:
# Rung=1 desc_dim=3 Features=40 'E_T': E/(kb*T), 'rho': density, 'AAV': AAV, 'PF': PF, 'VPA': VPA, 'NBI': NBI, 'SBI': SBI,  'RBI': RBI, 'WE': WE, 
        # 'B': B, 'N': N, 'S': S, 'O': O, 'F': F, 'Na': Na, 'Al': Al, 'Si': Si, 'P': P, 'Cl': Cl, 'Ge': Ge, 'Br': Br, 'I': I, 'cub_root_V': V^(1/3),
        # "Na-Na": data['Na-Na'], "Na-Na-max": data['Na-Na-max'], "Na-Na-avg": data['Na-Na-avg'], "bottleneck_min": data['bottleneck_min'], "bottleneck_avg": data['bottleneck_avg'],
        # "poly_dist_max": data['poly_dist_max'], "poly_dist_std": data['poly_dist_std'], "poly_dist_avg": data['poly_dist_avg'], "poly_dist_avg_fixed": data['poly_dist_avg_fixed'],
        # "face-area-max": data['face-area-max'], "face-area-avg": data['face-area-avg'], "face-area-min": data['face-area-min'], "face-area-count": data['face-area-count'],
        # "pockets-per-ion": data['pockets-per-ion'], "pockets-per-volume": data['pockets-per-volume'],
        # "paths-per-ion": data['paths-per-ion'], "paths-per-volume": data['paths-per-volume'],
# T = 900K
# time input_parsing: 0.0451138 s
# Time to generate feat space: 0.0210624 s
# Projection time: 0.0028353 s
# Time to get best features on rank : 8.51e-05 s
# |---Complete final combination/selection from all ranks: 0.00037 s
# Time for SIS: 0.0038822 s
# Time for l0-norm: 0.0048374 s
# Projection time: 0.0073381 s
# Time to get best features on rank : 7.62e-05 s
# |---Complete final combination/selection from all ranks: 0.0003026 s
# Time for SIS: 0.0092793 s
# Time for l0-norm: 0.0052008 s
# Projection time: 0.0049197 s
# Time to get best features on rank : 7.65e-05 s
# |---Complete final combination/selection from all ranks: 0.0002999 s
# Time for SIS: 0.0081385 s
# Time for l0-norm: 0.023731 s
# Train RMSE: 0.395729 Unitless
# c0 + a0 * (face-area-avg - O)

# Train RMSE: 0.316013 Unitless
# c0 + a0 * (Si + Al) + a1 * (face-area-avg - O)

# Train RMSE: 0.261622 Unitless
# c0 + a0 * (NBI - rho) + a1 * (Si + Al) + a2 * (face-area-avg - O)
# Coefficients
# Task   a0                      a1                      a2                      c0
# all ,  5.460781407147065e-01, -2.859471827321366e+00,  1.103983329123214e+00, -1.243742692679863e+01, 